# 03 — Lịch sử thuộc tính, cây danh mục và catalog item

Bước E của pipeline: chuẩn bị metadata **theo thời gian** để các bước sau tra cứu tại thời điểm query `t`.

- Đầu vào: `data/raw/item_properties_part1.csv`, `item_properties_part2.csv`, `category_tree.csv` (chỉ đọc), `events_sessionized.parquet` (notebook 02).
- Đầu ra:
  - `data/processed/item_property_history.parquet`: mọi cập nhật `categoryid` / `available`, sắp theo `itemid, timestamp`.
  - `data/processed/category_tree.parquet`: `categoryid, parentid (nullable), is_root, depth, root_categoryid`.
  - `data/processed/item_catalog.parquet`: thời điểm item được biết đến (`known_at_*`) và cờ có metadata.
  - `reports/metadata/metrics.json`.
- Đọc 20.275.902 dòng thuộc tính theo khối (`chunksize`); không nạp toàn bộ vào bộ nhớ.
- Mọi số liệu được so với kết quả khảo sát cũ; lệch thì notebook dừng để người dùng xem lại, không tự chỉnh cho khớp.

Hàm dùng chung nằm trong `src/rr/` (quyết định D6). Chạy bằng kernel `.venv`.

## 1. Đường dẫn, giao thức và kiểm tra đầu vào

SHA-256 của ba CSV phải khớp `configs/protocol.json → raw_files` (mã băm của bộ Kaggle gốc). `events_sessionized.parquet` phải khớp report notebook 02.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/rr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display
from rr.io import (content_sha256, environment, fingerprint, read_json, rel, require,
                   save_verified, utc_now, verify_inputs, write_json)

PROTOCOL_PATH = ROOT / "configs/protocol.json"
PROTOCOL = read_json(PROTOCOL_PATH)
RAW = ROOT / "data/raw"
PROPERTY_FILES = {1: RAW / "item_properties_part1.csv", 2: RAW / "item_properties_part2.csv"}
TREE_FILE = RAW / "category_tree.csv"
PROCESSED = ROOT / "data/processed"
REPORT_PATH = ROOT / "reports/metadata/metrics.json"
KEEP_PROPERTIES = PROTOCOL["metadata"]["properties"]
CHUNKSIZE = 2_000_000

EXPECTED = {  # khảo sát cũ, ghi trong README/docs/HIEN_TRANG.md
    "property_rows": 20_275_902, "kept_updates": 2_291_853, "events_items": 235_061,
    "events_items_without_metadata": 49_815, "categories": 1_669, "roots": 25,
    "categories_outside_tree": 30, "items_with_outside_categories": 137,
    "raw_events_before_first_property": 137_193,
}

raw_inputs = {rel(path, ROOT): {k: v for k, v in PROTOCOL["raw_files"][rel(path, ROOT)].items() if k != "rows"}
              for path in [*PROPERTY_FILES.values(), TREE_FILE, RAW / "events.csv"]}
inputs = verify_inputs(ROOT, raw_inputs)
split_report = read_json(ROOT / "reports/session_split/metrics.json")
sessionized_rel = "data/processed/events_sessionized.parquet"
inputs.update(verify_inputs(ROOT, {sessionized_rel: split_report["artifacts"][sessionized_rel]}))
print("Đầu vào khớp SHA-256:", *inputs, sep="\n  ")
print(environment())

## 2. Ví dụ biên cho các hàm của notebook

- `parse_tree()`: tìm gốc, độ sâu và phát hiện chu trình trong cây danh mục.
- `first_by_item()`: thời điểm nhỏ nhất theo item, bỏ qua giá trị thiếu.

In [ ]:
def parse_tree(tree):
    """Trả về depth và root cho mỗi categoryid; báo lỗi nếu có chu trình hoặc cha ngoài cây."""
    parent = {int(c): (None if pd.isna(p) else int(p)) for c, p in zip(tree["categoryid"], tree["parentid"])}
    outside = {p for p in parent.values() if p is not None and p not in parent}
    depth, root = {}, {}
    for start in parent:
        path, node = [], start
        while node not in depth:
            if node in path:
                raise ValueError(f"Chu trình trong cây danh mục tại {node}.")
            up = parent[node]
            if up is None or up in outside:
                depth[node], root[node] = 0, node
                break
            path.append(node)
            node = up
        for item in reversed(path):  # cha của mỗi phần tử đã có depth
            depth[item] = depth[parent[item]] + 1
            root[item] = root[parent[item]]
    return depth, root, outside


def first_by_item(itemids, timestamps):
    frame = pd.DataFrame({"itemid": itemids, "timestamp": timestamps}).dropna()
    return frame.groupby("itemid")["timestamp"].min().astype("int64")


_tree = pd.DataFrame({"categoryid": [1, 2, 3, 4], "parentid": pd.array([pd.NA, 1, 2, pd.NA], dtype="Int64")})
_depth, _root, _outside = parse_tree(_tree)
assert _depth == {1: 0, 2: 1, 3: 2, 4: 0} and _root == {1: 1, 2: 1, 3: 1, 4: 4} and not _outside
try:
    parse_tree(pd.DataFrame({"categoryid": [1, 2], "parentid": pd.array([2, 1], dtype="Int64")}))
    raise AssertionError("Phải phát hiện chu trình.")
except ValueError:
    pass
assert first_by_item([5, 5, 6], [30, 10, np.nan]).to_dict() == {5: 10}
del _tree, _depth, _root, _outside
print("Đã kiểm tra ví dụ biên: cây nhiều tầng, chu trình, thời điểm nhỏ nhất bỏ giá trị thiếu.")

## 3. Cây danh mục

`parentid` thiếu nghĩa là danh mục gốc; không điền 0. Kiểm tra trùng ID, cha tham chiếu ngoài cây và chu trình.

In [ ]:
tree_raw = pd.read_csv(TREE_FILE, dtype="string")
require(list(tree_raw.columns) == ["categoryid", "parentid"], f"Cột cây danh mục sai: {list(tree_raw.columns)}")
require(tree_raw["categoryid"].notna().all() and tree_raw["categoryid"].str.fullmatch(r"[0-9]+").all(), "categoryid không hợp lệ.")
require(tree_raw["parentid"].dropna().str.fullmatch(r"[0-9]+").all(), "parentid không hợp lệ.")
tree = pd.DataFrame({
    "categoryid": tree_raw["categoryid"].astype("int64"),
    "parentid": tree_raw["parentid"].astype("Int64"),
})
require(tree["categoryid"].is_unique, "categoryid trùng trong cây.")
depth, root, outside_parents = parse_tree(tree)
tree["is_root"] = tree["parentid"].isna()
tree["depth"] = tree["categoryid"].map(depth).astype("int16")
tree["root_categoryid"] = tree["categoryid"].map(root).astype("int64")
tree = tree.sort_values("categoryid").reset_index(drop=True)
tree_stats = {
    "categories": len(tree), "roots": int(tree["is_root"].sum()), "duplicate_ids": 0,
    "cycles": 0, "parents_outside_tree": len(outside_parents), "max_depth": int(tree["depth"].max()),
}
display(pd.Series(tree_stats, name="category_tree"))
display(tree.head())

## 4. Đọc hai phần thuộc tính theo khối

Mỗi khối được kiểm tra schema, đếm giá trị thiếu và băm từng dòng (`hash_pandas_object` trên 4 cột gốc) để phát hiện **trùng tuyệt đối xuyên hai tệp** mà không cần giữ 20 triệu dòng chuỗi. Chỉ giữ cập nhật `categoryid` / `available` cho pipeline chính, kèm `source_part` để truy vết.

Đồng thời ghi nhận: tập item có bất kỳ thuộc tính nào, thời điểm thuộc tính đầu tiên của từng item, khoảng thời gian.

In [ ]:
PROPERTY_COLUMNS = ["timestamp", "itemid", "property", "value"]
row_hashes, kept, item_first_property = [], [], []
property_stats = {}
for part, path in PROPERTY_FILES.items():
    stats = {"rows": 0, "missing": dict.fromkeys(PROPERTY_COLUMNS, 0), "invalid_timestamp": 0, "invalid_itemid": 0}
    for chunk in pd.read_csv(path, dtype="string", chunksize=CHUNKSIZE):
        require(list(chunk.columns) == PROPERTY_COLUMNS, f"Cột sai trong {path.name}: {list(chunk.columns)}")
        stats["rows"] += len(chunk)
        for column in PROPERTY_COLUMNS:
            stats["missing"][column] += int(chunk[column].isna().sum())
        stats["invalid_timestamp"] += int((~chunk["timestamp"].str.fullmatch(r"[0-9]{1,18}", na=False)).sum())
        stats["invalid_itemid"] += int((~chunk["itemid"].str.fullmatch(r"[0-9]{1,18}", na=False)).sum())
        row_hashes.append(pd.util.hash_pandas_object(chunk, index=False).to_numpy())
        timestamps = chunk["timestamp"].astype("int64")
        itemids = chunk["itemid"].astype("int64")
        item_first_property.append(first_by_item(itemids, timestamps))
        mask = chunk["property"].isin(KEEP_PROPERTIES)
        kept.append(pd.DataFrame({
            "itemid": itemids[mask].to_numpy(), "timestamp": timestamps[mask].to_numpy(),
            "property": chunk.loc[mask, "property"].astype(str).to_numpy(),
            "value": chunk.loc[mask, "value"].to_numpy(), "source_part": np.int8(part),
        }))
    property_stats[path.name] = stats
    print(path.name, f"{stats['rows']:,} dòng", stats["missing"])

hashes = np.concatenate(row_hashes)
del row_hashes
total_property_rows = len(hashes)
absolute_duplicates = int(total_property_rows - len(np.unique(hashes)))
del hashes
first_property = pd.concat(item_first_property).groupby(level=0).min().rename("first_property_ms")
del item_first_property
history = pd.concat(kept, ignore_index=True)
del kept
require(sum(s["invalid_timestamp"] + s["invalid_itemid"] for s in property_stats.values()) == 0, "Có timestamp/itemid thuộc tính không hợp lệ.")
require(sum(sum(s["missing"].values()) for s in property_stats.values()) == 0, "Thuộc tính có giá trị thiếu.")
print(f"Tổng dòng thuộc tính: {total_property_rows:,} | trùng tuyệt đối: {absolute_duplicates:,} | cập nhật giữ lại: {len(history):,}")

## 5. Lịch sử `categoryid` / `available`

- Không được có hai giá trị khác nhau cho cùng `(itemid, property, timestamp)` (xung đột).
- `available` chỉ nhận `0`/`1`; `categoryid` là số nguyên.
- Danh mục ngoài cây được giữ nguyên mã, đánh dấu chưa biết cha; không tự dựng quan hệ.

In [ ]:
history["property"] = pd.Categorical(history["property"], categories=KEEP_PROPERTIES)
history["value"] = history["value"].astype("string")
history = history.sort_values(["itemid", "timestamp", "property", "source_part"], kind="stable").reset_index(drop=True)
conflicts = int(history.groupby(["itemid", "property", "timestamp"], observed=True)["value"].nunique().gt(1).sum())
same_key_rows = int(history.duplicated(["itemid", "property", "timestamp"]).sum())
is_available = history["property"].eq("available")
require(history.loc[is_available, "value"].isin(["0", "1"]).all(), "available có giá trị khác 0/1.")
require(history.loc[~is_available, "value"].str.fullmatch(r"[0-9]+").all(), "categoryid không phải số nguyên.")

event_items = pd.read_parquet(PROCESSED / "events_sessionized.parquet", columns=["itemid", "timestamp", "split"])
event_item_set = set(event_items["itemid"].unique())
category_values = history.loc[~is_available, ["itemid", "value"]].assign(categoryid=lambda f: f["value"].astype("int64"))
outside_tree = category_values.loc[~category_values["categoryid"].isin(tree["categoryid"])]
outside_codes = int(outside_tree["categoryid"].nunique())
outside_items = set(outside_tree["itemid"].unique())
history_stats = {
    "kept_updates": len(history),
    "by_property": {str(k): int(v) for k, v in history["property"].value_counts().items()},
    "conflicts_same_item_property_timestamp": conflicts,
    "rows_sharing_item_property_timestamp": same_key_rows,
    "items_with_history": int(history["itemid"].nunique()),
    "categories_outside_tree": outside_codes,
    "items_with_outside_categories": len(outside_items),
    "items_with_outside_categories_in_events": len(outside_items & event_item_set),
    "timestamp_min_utc": pd.to_datetime(int(history["timestamp"].min()), unit="ms", utc=True).isoformat(),
    "timestamp_max_utc": pd.to_datetime(int(history["timestamp"].max()), unit="ms", utc=True).isoformat(),
}
display(pd.Series(history_stats, name="history"))
display(history.head())
del category_values, outside_tree

## 6. Catalog item và thời điểm được biết đến

`known_at` là thời điểm sớm nhất hệ thống có thể biết item tồn tại: sự kiện đầu tiên trong **dữ liệu dùng để fit** hoặc bản thuộc tính đầu tiên (bất kỳ thuộc tính nào).

| Cột | Ý nghĩa |
| --- | --- |
| `first_event_ms` | Sự kiện đầu tiên trong train/validation/test (không dùng `boundary_drop`) |
| `first_train_event_ms`, `first_trainval_event_ms` | Sự kiện đầu tiên trong train; trong train + validation |
| `first_property_ms` | Bản thuộc tính đầu tiên của item |
| `known_at_ms` | `min(first_event_ms, first_property_ms)` |
| `known_at_train_ms`, `known_at_trainval_ms` | Bản dùng lọc ứng viên khi fit trên train / train+validation (`configs/protocol.json → candidates.known_at`) |
| `has_metadata` | Item có ít nhất một bản thuộc tính |

In [ ]:
usable = event_items[event_items["split"].astype(str) != "boundary_drop"]
def first_event(mask):
    return usable.loc[mask].groupby("itemid")["timestamp"].min()

split_text = usable["split"].astype(str)
first_events = pd.DataFrame({
    "first_event_ms": first_event(slice(None)),
    "first_train_event_ms": first_event(split_text.eq("train")),
    "first_trainval_event_ms": first_event(split_text.isin(["train", "validation"])),
})
catalog_items = np.union1d(np.union1d(event_items["itemid"].unique(), first_property.index.to_numpy()), [])
catalog = pd.DataFrame({"itemid": catalog_items.astype("int64")}).set_index("itemid")
catalog = catalog.join(first_events).join(first_property)
for column in catalog.columns:
    catalog[column] = catalog[column].astype("Int64")
def nullable_min(a, b):
    return pd.concat([a, b], axis=1).min(axis=1, skipna=True).astype("Int64")
catalog["known_at_ms"] = nullable_min(catalog["first_event_ms"], catalog["first_property_ms"])
catalog["known_at_train_ms"] = nullable_min(catalog["first_train_event_ms"], catalog["first_property_ms"])
catalog["known_at_trainval_ms"] = nullable_min(catalog["first_trainval_event_ms"], catalog["first_property_ms"])
catalog["has_metadata"] = catalog["first_property_ms"].notna()
history_items = history.groupby(["itemid", "property"], observed=True).size().unstack(fill_value=0)
catalog["has_category"] = catalog.index.isin(history_items.index[history_items["categoryid"] > 0])
catalog["has_available"] = catalog.index.isin(history_items.index[history_items["available"] > 0])
catalog["in_events"] = catalog.index.isin(list(event_item_set))
catalog = catalog.reset_index()

raw_event_ts = pd.read_csv(RAW / "events.csv", usecols=["timestamp"], dtype={"timestamp": "int64"})["timestamp"]
global_first_property = int(history["timestamp"].min()) if len(history) else None
global_first_any_property = int(first_property.min())
events_by_item_first = usable.join(first_property, on="itemid")
catalog_stats = {
    "catalog_items": len(catalog),
    "events_items": int(catalog["in_events"].sum()),
    "events_items_without_metadata": int((catalog["in_events"] & ~catalog["has_metadata"]).sum()),
    "property_only_items": int((~catalog["in_events"]).sum()),
    "items_without_known_at": int(catalog["known_at_ms"].isna().sum()),
    "first_property_utc": pd.to_datetime(global_first_any_property, unit="ms", utc=True).isoformat(),
    "raw_events_before_first_property": int(raw_event_ts.lt(global_first_any_property).sum()),
    "usable_events_before_item_first_property": int(
        (events_by_item_first["first_property_ms"].isna()
         | events_by_item_first["timestamp"].lt(events_by_item_first["first_property_ms"])).sum()),
}
display(pd.Series(catalog_stats, name="catalog"))
display(catalog.head())
del raw_event_ts, events_by_item_first, usable

## 7. Đối chiếu với khảo sát cũ

Mọi tiêu chí của T08 phải khớp. Nếu lệch, notebook dừng **trước khi ghi đầu ra**.

In [ ]:
observed = {
    "property_rows": total_property_rows, "kept_updates": len(history),
    "events_items": catalog_stats["events_items"],
    "events_items_without_metadata": catalog_stats["events_items_without_metadata"],
    "categories": tree_stats["categories"], "roots": tree_stats["roots"],
    "categories_outside_tree": outside_codes, "items_with_outside_categories": len(outside_items),
    "raw_events_before_first_property": catalog_stats["raw_events_before_first_property"],
}
comparison = pd.DataFrame({"expected": EXPECTED, "observed": observed})
comparison["match"] = comparison["expected"].eq(comparison["observed"])
display(comparison)
require(comparison["match"].all(), "Số liệu lệch khảo sát cũ; dừng để người dùng xem lại.")
require(absolute_duplicates == 0, "Có dòng thuộc tính trùng tuyệt đối.")
require(conflicts == 0, "Có xung đột cùng (item, property, timestamp).")
require(history_stats["items_with_outside_categories_in_events"] == 0, "Item có danh mục ngoài cây xuất hiện trong events.")
require(tree_stats["parents_outside_tree"] == 0, "Có parentid tham chiếu ngoài cây.")
print("Mọi tiêu chí khớp khảo sát cũ.")

## 8. Lưu Parquet đã kiểm tra và báo cáo

In [ ]:
artifacts = {}
for frame, name, key in [
    (history, "item_property_history.parquet", ["itemid", "timestamp", "property", "source_part"]),
    (tree, "category_tree.parquet", ["categoryid"]),
    (catalog, "item_catalog.parquet", ["itemid"]),
]:
    info = save_verified(frame, PROCESSED / name, key, ROOT)
    artifacts[info.pop("path")] = info
    print(f"Đã lưu {name}: {len(frame):,} dòng")

require(all(fingerprint(ROOT / path) == {k: v for k, v in info.items()} for path, info in inputs.items()),
        "Đầu vào thay đổi trong lúc chạy.")
report = {
    "generated_at_utc": utc_now(), **environment(),
    "notebook": "notebooks/03_lich_su_thuoc_tinh_va_catalog.ipynb",
    "protocol": {"path": rel(PROTOCOL_PATH, ROOT), "version": PROTOCOL["version"], **fingerprint(PROTOCOL_PATH)},
    "inputs": inputs,
    "property_files": property_stats,
    "property_rows": total_property_rows,
    "absolute_duplicate_rows_across_files": absolute_duplicates,
    "category_tree": tree_stats,
    "history": history_stats,
    "catalog": catalog_stats,
    "expected_vs_observed": {k: {"expected": EXPECTED[k], "observed": observed[k]} for k in EXPECTED},
    "known_at_rule": PROTOCOL["candidates"]["known_at"],
    "artifacts": artifacts,
    "status": "PASS",
}
write_json(REPORT_PATH, report)
print("Báo cáo:", rel(REPORT_PATH, ROOT))